# `gold.dim_management_group` — load

One row per house. Sourced from `gold.dim_ticker` so the key is built from exactly the
string the dimension carries — the fact builds `management_group_key` the same way, which is
what makes the two agree without a second copy of the labelling rules.

`NoInfo` and `NotApplicable` are kept as rows. They are real answers to "which house runs
this?", and dropping them would make the group counts stop adding up to the universe.

In [ ]:
CREATE OR REPLACE TEMP VIEW gold_stage_dim_management_group AS
SELECT MD5(management_group)  AS management_group_key,
       management_group,
       COUNT(DISTINCT ticker) AS trusts_managed
FROM `index-vs-trust-pipeline`.gold.dim_ticker
GROUP BY management_group;

In [ ]:
MERGE INTO `index-vs-trust-pipeline`.gold.dim_management_group AS t
USING gold_stage_dim_management_group AS s
   ON t.management_group_key = s.management_group_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

Expected: **53** rows — 51 real houses plus `NoInfo` and `NotApplicable` — **0** duplicate
keys, and `trusts_managed` summing to **102**, the whole dimension.

In [ ]:
SELECT COUNT(*)                                     AS rows_total,
       COUNT(*) - COUNT(DISTINCT management_group_key) AS duplicate_keys,
       SUM(trusts_managed)                          AS trusts_covered,
       MAX(trusts_managed)                          AS largest_house
FROM `index-vs-trust-pipeline`.gold.dim_management_group;